In [1]:
import random
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from threading import Thread
from transformers import TextIteratorStreamer

modelPath = '../15-transformers-basics/models/Qwen3-0.6B'

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(modelPath)
model = AutoModelForCausalLM.from_pretrained(
    modelPath,
    dtype=torch.bfloat16,           # 与模型保持一致，使用 bfloat16
    device_map="auto",              # 自动放到 GPU 上
)
model.eval()

list_history = []


def chat(message, history):
    """聊天函数：接收用户消息和对话历史，流式返回模型回复"""
    # 构建对话消息列表，先加入系统提示词
    messages = [{"role": "system", "content": "你是一个有帮助的AI助手，请用中文回答用户的问题。"}]
    # 将历史对话加入消息列表
    for msg in history:
        messages.append(msg)
    messages.append({"role": "user", "content": message})
    
    # 应用聊天模板，将消息列表转为模型输入文本
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # 添加 assistant 开头，引导模型生成回复
        enable_thinking=False,        # 关闭思考模式，直接给出答案
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    # 创建流式输出器，逐 token 输出结果
    streamer = TextIteratorStreamer(
        tokenizer,
        skip_prompt=True,          # 跳过输入部分，只输出新生成的内容
        skip_special_tokens=True,  # 跳过特殊标记如 <|im_end|>
    )
    
    # 生成参数
    generation_kwargs = dict(
        model_inputs,
        streamer=streamer,
        max_new_tokens=1024,       # 最大生成 token 数
        do_sample=True,            # 采样模式，增加回复多样性
        temperature=0.6,           # 温度：越高越随机，越低越确定
        top_p=0.95,               # 核采样：保留概率前 95% 的 token
        top_k=20,                 # Top-K 采样：只从概率最高的 20 个 token 中选
    )
    
    # 在单独的线程中运行生成，避免阻塞主线程的流式输出
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()
    
    # 流式返回生成结果，每生成一个 token 就更新一次界面
    response = ""
    for new_text in streamer:
        response += new_text
        yield response


demo = gr.ChatInterface(
    fn=chat,
    title="Qwen3-0.6B 聊天机器人",
)

demo.launch()

S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 377.69it/s]


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


S:\PythonAILearningCourse\15-transformers-basics\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'ChatGLMConfig' object has no attribute 'max_length'